# Stuff+ Quick Start
Walk through each pipeline step interactively.

In [ ]:
import sys; sys.path.insert(0, '..')   # so imports resolve from project root
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

## 1. Fetch & Cache Data
Fetches one season and caches it as Parquet. Subsequent runs load from disk instantly.

In [ ]:
from src.data.fetch import fetch_season, cache_summary

# Fetch a single season (~5-10 min first time, instant after)
df_raw = fetch_season(2023)
print(f'Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}')
df_raw.head(3)

In [ ]:
# Check what's in the cache
cache_summary()

## 2. Preprocess

In [ ]:
from src.data.preprocess import clean, save_processed

df_clean = clean(df_raw)
print(f'After cleaning: {len(df_clean):,} rows')
print(df_clean['pitch_group'].value_counts())

## 3. Feature Engineering

In [ ]:
from src.features.engineer import build_features

df_feat = build_features(df_clean)
print('New columns:', [c for c in df_feat.columns if c not in df_clean.columns])
df_feat[['player_name', 'pitch_type', 'release_speed', 'vaa', 'haa',
          'induced_vertical_break', 'arm_side_break']].head(10)

## 4. Train (single-season demo)

In [ ]:
# Save features so train.py can find them
import pyarrow as pa, pyarrow.parquet as pq, yaml

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

out = f"{cfg['paths']['processed_data']}/statcast_features.parquet"
pq.write_table(pa.Table.from_pandas(df_feat), out)
print('Saved to', out)

In [ ]:
from src.models.train import run_training

# Train all three pitch groups, hold out 2023 as val
results = run_training(val_season=2023)
for group, r in results.items():
    print(f"{group:10s}  RMSE={r['metrics']['rmse']:.4f}  Corr={r['metrics']['corr']:.4f}")

## 5. Generate Stuff+ Scores & Leaderboard

In [ ]:
from src.models.evaluate import predict_stuff_plus, build_leaderboard, plot_stuff_plus_distribution

df_scored = predict_stuff_plus(df_feat)
lb = build_leaderboard(df_scored, min_pitches=100)
lb.head(25)

In [ ]:
plot_stuff_plus_distribution(df_scored, save=False)